# Hands-on 1: Precipitation Forecast Post-Processing

**Module 2 — Deep Learning Best Practices & Applications**  
CIMA Foundation PhD Course, 27 May 2026

In this notebook we build a simple CNN to post-process (bias-correct) NWP precipitation forecasts.  
We use the **Modern ML Stack**: Zarr + xarray for data, PyTorch Lightning for training, Hydra/OmegaConf for configuration, and MLFlow for experiment tracking.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/meteocima/Monaco_DLCourse/blob/main/notebooks/01_precipitation_postprocessing.ipynb)

## 0. Setup

We use [**uv**](https://docs.astral.sh/uv/) as our Python package manager — it's a fast, modern replacement for pip + virtualenv.

| Context | Command | What it does |
|---------|---------|-------------|
| **New project** | `uv init` | Scaffold a project with `pyproject.toml` |
| **Add a dependency** | `uv add torch xarray` | Add packages to `pyproject.toml` and install them |
| **Install everything** | `uv sync` | Create a `.venv` and install from the lockfile |
| **Run a command** | `uv run jupyter lab` | Run inside the managed `.venv` |
| **Colab / system Python** | `uv pip install --system -r requirements.txt` | Install without a `.venv` (for environments you don't manage) |

On **Colab** there is no `.venv` — Google provides the Python environment — so we use `uv pip install --system`.

In [ ]:
import os

# On Colab: clone the repo and install dependencies
if "COLAB_RELEASE_TAG" in os.environ:
    !git clone https://github.com/meteocima/Monaco_DLCourse.git /content/Monaco_DLCourse 2>/dev/null || true
    %cd /content/Monaco_DLCourse/notebooks
    !pip install -q uv
    !uv pip install --system -q -r ../requirements.txt

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import xarray as xr
import zarr
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import pytorch_lightning as pl
from omegaconf import OmegaConf
import mlflow

print(f"PyTorch version: {torch.__version__}")
print(f"Lightning version: {pl.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

## 1. Configuration with Hydra / OmegaConf

In production, Hydra loads YAML config files and composes them automatically via its `@hydra.main` decorator. In a notebook we use **OmegaConf** (Hydra's config backend) to load the same YAML files directly — giving us structured, typed configuration without needing the CLI.

The config lives in `configs/precipitation.yaml`. You can override any value inline after loading.

In [ ]:
# Load config from YAML (same file Hydra would use in a script)
cfg = OmegaConf.load("../configs/precipitation.yaml")

# You can override values inline, e.g.:
# cfg.training.max_epochs = 50
# cfg.data.batch_size = 64

print(OmegaConf.to_yaml(cfg))

## 2. Data Loading with Zarr + xarray

We load **real ERA5 precipitation** from the [WeatherBench 2](https://weatherbench2.readthedocs.io/) cloud Zarr store (no download needed!).  
The dataset lives on Google Cloud Storage and is opened lazily — only the subset we select gets transferred.

To simulate the post-processing task, we use ERA5 as **truth** and create a synthetic **NWP forecast** by adding systematic bias and noise. In practice, the forecast would come from a separate NWP model (e.g., IFS HRES).

In [ ]:
# Open the WeatherBench 2 ERA5 Zarr store (lazy — no data downloaded yet)
ds_era5 = xr.open_zarr(cfg.data.zarr_url)
print("Available variables:", list(ds_era5.data_vars))
print(ds_era5)

In [ ]:
# Select precipitation, subset time and latitude, and load into memory
from scipy.ndimage import gaussian_filter

precip = (
    ds_era5[cfg.data.variable]
    .sel(time=slice(cfg.data.time_start, cfg.data.time_end))
    .isel(latitude=slice(0, cfg.data.lat_slice))  # 32 latitudes → 32×32 grid
)
print(f"Selected shape (time, lat, lon): {dict(precip.sizes)}")

# Load into memory (small subset — should take a few seconds)
truth = precip.values.astype(np.float32)
print(f"Loaded truth array: {truth.shape} — {truth.nbytes / 1e6:.1f} MB")

# Create synthetic NWP forecast: ERA5 truth + systematic bias + noise
rng = np.random.default_rng(42)
bias = cfg.data.bias * np.ones(truth.shape[1:], dtype=np.float32)
noise = rng.normal(0, cfg.data.noise_std, size=truth.shape).astype(np.float32)
forecast = np.clip(truth + bias + noise, 0, None)

print(f"Forecast shape: {forecast.shape}, Truth shape: {truth.shape}")
print(f"Mean bias: {(forecast - truth).mean():.3f} mm")

# Wrap in an xarray Dataset for exploration
ds = xr.Dataset({
    "forecast": (["time", "lat", "lon"], forecast),
    "truth": (["time", "lat", "lon"], truth),
})
print(ds)

In [ ]:
# Visualize a sample
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
idx = 0

im0 = axes[0].imshow(forecast[idx], cmap="Blues", vmin=0)
axes[0].set_title("NWP Forecast")
plt.colorbar(im0, ax=axes[0], fraction=0.046)

im1 = axes[1].imshow(truth[idx], cmap="Blues", vmin=0)
axes[1].set_title("ERA5 Truth")
plt.colorbar(im1, ax=axes[1], fraction=0.046)

im2 = axes[2].imshow(forecast[idx] - truth[idx], cmap="RdBu_r")
axes[2].set_title("Forecast Bias")
plt.colorbar(im2, ax=axes[2], fraction=0.046)

plt.suptitle("Sample Precipitation Fields (mm)")
plt.tight_layout()
plt.show()

## 3. PyTorch Dataset & DataModule

In [ ]:
class PrecipitationDataset(Dataset):
    """Dataset for precipitation post-processing."""

    def __init__(self, forecast, truth):
        self.forecast = torch.from_numpy(forecast).unsqueeze(1)  # (N, 1, H, W)
        self.truth = torch.from_numpy(truth).unsqueeze(1)

    def __len__(self):
        return len(self.forecast)

    def __getitem__(self, idx):
        return self.forecast[idx], self.truth[idx]


class PrecipDataModule(pl.LightningDataModule):
    """Lightning DataModule for precipitation data."""

    def __init__(self, forecast, truth, cfg):
        super().__init__()
        self.forecast = forecast
        self.truth = truth
        self.cfg = cfg

    def setup(self, stage=None):
        n = len(self.forecast)
        n_train = int(n * self.cfg.data.train_split)
        self.train_ds = PrecipitationDataset(
            self.forecast[:n_train], self.truth[:n_train]
        )
        self.val_ds = PrecipitationDataset(
            self.forecast[n_train:], self.truth[n_train:]
        )

    def train_dataloader(self):
        return DataLoader(
            self.train_ds, batch_size=self.cfg.data.batch_size, shuffle=True
        )

    def val_dataloader(self):
        return DataLoader(
            self.val_ds, batch_size=self.cfg.data.batch_size
        )


dm = PrecipDataModule(forecast, truth, cfg)
dm.setup()
print(f"Train samples: {len(dm.train_ds)}, Val samples: {len(dm.val_ds)}")

## 4. Model: Simple CNN for Post-Processing

A residual CNN that learns to correct the forecast. The model predicts a *correction field* that is added to the input forecast.

In [ ]:
class PrecipPostProcessor(pl.LightningModule):
    """CNN that learns a residual correction for precipitation forecasts."""

    def __init__(self, cfg):
        super().__init__()
        self.save_hyperparameters()
        self.cfg = cfg

        layers = []
        in_ch = cfg.model.in_channels
        for i in range(cfg.model.n_layers):
            out_ch = cfg.model.hidden_channels if i < cfg.model.n_layers - 1 else cfg.model.out_channels
            layers.append(nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1))
            if i < cfg.model.n_layers - 1:
                layers.append(nn.ReLU())
            in_ch = out_ch
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        correction = self.net(x)
        return x + correction  # residual connection

    def training_step(self, batch, batch_idx):
        forecast, truth = batch
        pred = self(forecast)
        loss = F.mse_loss(pred, truth)
        self.log("train_loss", loss, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        forecast, truth = batch
        pred = self(forecast)
        loss = F.mse_loss(pred, truth)
        mae = F.l1_loss(pred, truth)
        self.log("val_loss", loss, prog_bar=True)
        self.log("val_mae", mae)
        return loss

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.cfg.training.learning_rate)


model = PrecipPostProcessor(cfg)
print(model)

## 5. Training with Lightning + MLFlow Tracking

In [ ]:
# Set up MLFlow experiment
mlflow.set_experiment(cfg.mlflow.experiment_name)

with mlflow.start_run(run_name="cnn_postprocessor_v1"):
    # Log configuration
    mlflow.log_params(OmegaConf.to_container(cfg, resolve=True))

    # Create trainer
    trainer = pl.Trainer(
        max_epochs=cfg.training.max_epochs,
        accelerator=cfg.training.accelerator,
        enable_progress_bar=True,
        log_every_n_steps=5,
    )

    # Train
    trainer.fit(model, dm)

    # Log final metrics
    val_loss = trainer.callback_metrics.get("val_loss")
    val_mae = trainer.callback_metrics.get("val_mae")
    if val_loss is not None:
        mlflow.log_metric("final_val_loss", val_loss.item())
    if val_mae is not None:
        mlflow.log_metric("final_val_mae", val_mae.item())

    print(f"\nFinal val_loss: {val_loss}")
    print(f"Final val_mae: {val_mae}")

## 6. Evaluation & Visualization

In [ ]:
# Run predictions on validation set
model.eval()
val_forecast = dm.val_ds.forecast
val_truth = dm.val_ds.truth

with torch.no_grad():
    val_pred = model(val_forecast)

# Convert to numpy for plotting
val_forecast_np = val_forecast.squeeze(1).numpy()
val_truth_np = val_truth.squeeze(1).numpy()
val_pred_np = val_pred.squeeze(1).numpy()

In [ ]:
# Compare: raw forecast vs post-processed vs truth
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for row, idx in enumerate([0, 5]):
    vmin = min(val_truth_np[idx].min(), val_forecast_np[idx].min())
    vmax = max(val_truth_np[idx].max(), val_forecast_np[idx].max())

    im0 = axes[row, 0].imshow(val_forecast_np[idx], cmap="Blues", vmin=vmin, vmax=vmax)
    axes[row, 0].set_title("Raw NWP Forecast")
    plt.colorbar(im0, ax=axes[row, 0], fraction=0.046)

    im1 = axes[row, 1].imshow(val_pred_np[idx], cmap="Blues", vmin=vmin, vmax=vmax)
    axes[row, 1].set_title("Post-Processed (CNN)")
    plt.colorbar(im1, ax=axes[row, 1], fraction=0.046)

    im2 = axes[row, 2].imshow(val_truth_np[idx], cmap="Blues", vmin=vmin, vmax=vmax)
    axes[row, 2].set_title("ERA5 Truth")
    plt.colorbar(im2, ax=axes[row, 2], fraction=0.046)

plt.suptitle("Post-Processing Results", fontsize=14)
plt.tight_layout()
plt.show()

# Quantitative comparison
raw_mse = np.mean((val_forecast_np - val_truth_np) ** 2)
pp_mse = np.mean((val_pred_np - val_truth_np) ** 2)
print(f"Raw forecast MSE:     {raw_mse:.4f}")
print(f"Post-processed MSE:   {pp_mse:.4f}")
print(f"Improvement:          {(1 - pp_mse/raw_mse) * 100:.1f}%")

## 7. Exercises

1. **Change the architecture**: Try adding more layers or using a U-Net structure with skip connections
2. **Experiment with Hydra configs**: Modify `cfg` values (bias, noise, grid size) and observe how training changes
3. **Compare optimizers**: Try `SGD`, `AdamW`, or add a learning rate scheduler
4. **Add more input channels**: Load additional ERA5 variables from the Zarr store (e.g., `temperature`, `geopotential`) as extra CNN input channels
5. **Try a different region**: Change `lat_slice` to select different latitude bands and see how precipitation patterns differ
6. **Check MLFlow**: Run `!mlflow ui` and open the link to explore logged experiments